# Llama-Index Text-To-SQL

In [1]:
%%capture
!pip install duckdb duckdb-engine llama-index

In [2]:
from llama_index.core import SQLDatabase, SimpleDirectoryReader, Document
from llama_index.core.indices.struct_store import (
    NLSQLTableQueryEngine,
    SQLTableRetrieverQueryEngine,
)

In [3]:
from IPython.display import Markdown, display

## Basic Text-to-SQL with `NLSQLTableQueryEngine`


In [4]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
    column,
)

In [5]:
# creates a SQLAlchemy engine object that connects to an in-memory DuckDB database.
engine = create_engine("duckdb:///:memory:") # https://duckdb.org/
metadata_obj = MetaData()

In [6]:
# create city SQL table
table_name = "city_stats"
city_stats_table = Table(
    table_name,
    metadata_obj,
    Column("city_name", String(16), primary_key=True),
    Column("population", Integer),
    Column("country", String(16), nullable=False),
)

metadata_obj.create_all(engine)

In [7]:
# print tables
metadata_obj.tables.keys()

dict_keys(['city_stats'])

We introduce some test data into the `city_stats` table

In [8]:
from sqlalchemy import insert

rows = [
    {"city_name": "Toronto", "population": 2930000, "country": "Canada"},
    {"city_name": "Tokyo", "population": 13960000, "country": "Japan"},
    {"city_name": "Chicago", "population": 2679000, "country": "United States"},
    {"city_name": "Seoul", "population": 9776000, "country": "South Korea"},
]
for row in rows:
    stmt = insert(city_stats_table).values(**row)
    with engine.begin() as connection:
        cursor = connection.execute(stmt)

In [9]:
with engine.connect() as connection:
    cursor = connection.exec_driver_sql("SELECT * FROM city_stats")
    print(cursor.fetchall())

[('Toronto', 2930000, 'Canada'), ('Tokyo', 13960000, 'Japan'), ('Chicago', 2679000, 'United States'), ('Seoul', 9776000, 'South Korea')]


### Create SQLDatabase Object

In [10]:
from llama_index.core import SQLDatabase

In [11]:
sql_database = SQLDatabase(engine, include_tables=["city_stats"])

/usr/local/lib/python3.10/dist-packages/duckdb_engine/__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


### Query Index
- An Index is a data structure that allows us to quickly retrieve relevant context for a user query
- We are going to use the `NLSQLTableQueryEngine` as an query engine and run queries against it.

#### Using OpenAI model

In [12]:
import os
# https://platform.openai.com/account/api-keys
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

In [13]:
!pip install llama-index-llms-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.0 MB/s eta 0:00:00


In [14]:
from google.colab import userdata


In [16]:
#define LLM
from llama_index.core import ServiceContext, set_global_service_context

from llama_index.llms.cohere import Cohere
cohere_api_key = userdata.get('cohere_api_key')
# Initialize Cohere
llm = Cohere(api_key=cohere_api_key)

# configure service context
# service_context = ServiceContext.from_defaults(llm=llm)

In [17]:
#query_engine_openai = NLSQLTableQueryEngine(sql_database)
query_engine_openai = NLSQLTableQueryEngine(sql_database, llm=llm )

In [18]:
response = query_engine_openai.query("Which city has the highest population?")

In [19]:
response.response

'Tokyo has the highest population among all the cities in the dataset.'

In [20]:
response.metadata

{'bfcf1caa-b9f9-472a-b584-6d6f466e92b0': {'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;',
  'result': [('Tokyo',)],
  'col_keys': ['city_name']},
 'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;',
 'result': [('Tokyo',)],
 'col_keys': ['city_name']}

In [21]:
response_with_population = query_engine_openai.query("Which city has the highest population. Also provide the population?")

In [22]:
response_with_population.response

'Tokyo has the highest population among all the cities in the database, standing at an impressive 13.96 million people.'

## Advanced Text-to-SQL with `SQLTableRetrieverQueryEngine`

- Let's assume that you have a large number of tables in your database, and putting all the table schemas into the prompt may overflow the text-to-SQL prompt.

- We first index the schemas with our ObjectIndex, and then use our SQLTableRetrieverQueryEngine abstraction on top.

In [23]:
# creates a SQLAlchemy engine object that connects to an in-memory DuckDB database.
engine = create_engine("duckdb:///:memory:")
metadata_obj = MetaData()

In [24]:
# create city_stats SQL table
table_name = "city_stats"
city_stats_table = Table(
    table_name,
    metadata_obj,
    Column("city_name", String(16), primary_key=True),
    Column("population", Integer),
    Column("country", String(16), nullable=False),
)

all_table_names = ["city_stats"]

# create a ton of dummy tables
n = 100
for i in range(n):
    tmp_table_name = f"tmp_table_{i}"
    tmp_table = Table(
        tmp_table_name,
        metadata_obj,
        Column(f"tmp_field_{i}_1", String(16), primary_key=True),
        Column(f"tmp_field_{i}_2", Integer),
        Column(f"tmp_field_{i}_3", String(16), nullable=False),
    )
    all_table_names.append(f"tmp_table_{i}")

metadata_obj.create_all(engine)

In [25]:
all_table_names

['city_stats',
 'tmp_table_0',
 'tmp_table_1',
 'tmp_table_2',
 'tmp_table_3',
 'tmp_table_4',
 'tmp_table_5',
 'tmp_table_6',
 'tmp_table_7',
 'tmp_table_8',
 'tmp_table_9',
 'tmp_table_10',
 'tmp_table_11',
 'tmp_table_12',
 'tmp_table_13',
 'tmp_table_14',
 'tmp_table_15',
 'tmp_table_16',
 'tmp_table_17',
 'tmp_table_18',
 'tmp_table_19',
 'tmp_table_20',
 'tmp_table_21',
 'tmp_table_22',
 'tmp_table_23',
 'tmp_table_24',
 'tmp_table_25',
 'tmp_table_26',
 'tmp_table_27',
 'tmp_table_28',
 'tmp_table_29',
 'tmp_table_30',
 'tmp_table_31',
 'tmp_table_32',
 'tmp_table_33',
 'tmp_table_34',
 'tmp_table_35',
 'tmp_table_36',
 'tmp_table_37',
 'tmp_table_38',
 'tmp_table_39',
 'tmp_table_40',
 'tmp_table_41',
 'tmp_table_42',
 'tmp_table_43',
 'tmp_table_44',
 'tmp_table_45',
 'tmp_table_46',
 'tmp_table_47',
 'tmp_table_48',
 'tmp_table_49',
 'tmp_table_50',
 'tmp_table_51',
 'tmp_table_52',
 'tmp_table_53',
 'tmp_table_54',
 'tmp_table_55',
 'tmp_table_56',
 'tmp_table_57',
 'tmp_tab

In [26]:
with engine.connect() as connection:
    cursor = connection.exec_driver_sql("SELECT * FROM city_stats")
    print(cursor.fetchall())

[]


In [27]:
# insert dummy data
from sqlalchemy import insert

rows = [
    {"city_name": "Toronto", "population": 2930000, "country": "Canada"},
    {"city_name": "Tokyo", "population": 13960000, "country": "Japan"},
    {"city_name": "Chicago", "population": 2679000, "country": "United States"},
    {"city_name": "Seoul", "population": 9776000, "country": "South Korea"},
]
for row in rows:
    stmt = insert(city_stats_table).values(**row)
    with engine.begin() as connection:
        cursor = connection.execute(stmt)

In [28]:
with engine.connect() as connection:
    cursor = connection.exec_driver_sql("SELECT * FROM city_stats")
    print(cursor.fetchall())

[('Toronto', 2930000, 'Canada'), ('Tokyo', 13960000, 'Japan'), ('Chicago', 2679000, 'United States'), ('Seoul', 9776000, 'South Korea')]


In [29]:
with engine.connect() as connection:
    cursor = connection.exec_driver_sql("SELECT * FROM tmp_table_99")
    print(cursor.fetchall())

[]


In [30]:
sql_database = SQLDatabase(engine, include_tables=["city_stats"])

/usr/local/lib/python3.10/dist-packages/duckdb_engine/__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


### Construct Object Index

In [31]:
from llama_index.core.indices.struct_store import SQLTableRetrieverQueryEngine
from llama_index.core.objects import SQLTableNodeMapping, ObjectIndex, SQLTableSchema
from llama_index.core import VectorStoreIndex

In [32]:
%pip install llama-index-llms-cohere
%pip install llama-index-embeddings-cohere

In [33]:
table_node_mapping = SQLTableNodeMapping(sql_database)

table_schema_objs = []
for table_name in all_table_names:
    table_schema_objs.append(SQLTableSchema(table_name=table_name))
from llama_index.embeddings.cohere import CohereEmbedding

# with input_typ='search_query'
embed_model = CohereEmbedding(
    api_key=cohere_api_key,
    model_name="embed-english-v3.0",
    input_type="search_query",
)
obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_mapping,
    VectorStoreIndex,
    embed_model=embed_model,
)

In [34]:
table_schema_objs

[SQLTableSchema(table_name='city_stats', context_str=None),
 SQLTableSchema(table_name='tmp_table_0', context_str=None),
 SQLTableSchema(table_name='tmp_table_1', context_str=None),
 SQLTableSchema(table_name='tmp_table_2', context_str=None),
 SQLTableSchema(table_name='tmp_table_3', context_str=None),
 SQLTableSchema(table_name='tmp_table_4', context_str=None),
 SQLTableSchema(table_name='tmp_table_5', context_str=None),
 SQLTableSchema(table_name='tmp_table_6', context_str=None),
 SQLTableSchema(table_name='tmp_table_7', context_str=None),
 SQLTableSchema(table_name='tmp_table_8', context_str=None),
 SQLTableSchema(table_name='tmp_table_9', context_str=None),
 SQLTableSchema(table_name='tmp_table_10', context_str=None),
 SQLTableSchema(table_name='tmp_table_11', context_str=None),
 SQLTableSchema(table_name='tmp_table_12', context_str=None),
 SQLTableSchema(table_name='tmp_table_13', context_str=None),
 SQLTableSchema(table_name='tmp_table_14', context_str=None),
 SQLTableSchema(tabl

### Query Index with `SQLTableRetrieverQueryEngine`


In [35]:
query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_index.as_retriever(similarity_top_k=1),
    llm=llm
)

In [36]:
response = query_engine.query("Which city has the highest population?")

In [37]:
response

Response(response='Tokyo has the highest population among all the cities in the dataset.', source_nodes=[NodeWithScore(node=TextNode(id_='042b8628-9c6b-4fed-ad1e-e899750017b1', embedding=None, metadata={'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;', 'result': [('Tokyo',)], 'col_keys': ['city_name']}, excluded_embed_metadata_keys=['sql_query', 'result', 'col_keys'], excluded_llm_metadata_keys=['sql_query', 'result', 'col_keys'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text="[('Tokyo',)]", mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=None)], metadata={'042b8628-9c6b-4fed-ad1e-e899750017b1': {'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;', 'result': [('Tokyo',)], 'col_keys': ['city_name']}, 'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;', 'resul

In [38]:
response.response

'Tokyo has the highest population among all the cities in the dataset.'

In [39]:
response.metadata

{'042b8628-9c6b-4fed-ad1e-e899750017b1': {'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;',
  'result': [('Tokyo',)],
  'col_keys': ['city_name']},
 'sql_query': 'SELECT city_name\nFROM city_stats\nORDER BY population DESC\nLIMIT 1;',
 'result': [('Tokyo',)],
 'col_keys': ['city_name']}